# AVM v0 - train trên Google Colab (không có Test B)

Chạy lần lượt từng ô từ trên xuống: **fetch → crosswalk → clean → split → features → baseline → train → evaluate → tải kết quả về**.

- Dữ liệu: HF `tinixai/vietnam-real-estates` (license **CC BY-NC 4.0**) → mọi artifact là `NON_PRODUCTION`, **không** chia sẻ công khai / không dùng thương mại.
- **Test B (dữ liệu V-Nexus DB) bị bỏ qua**: notebook không đặt biến `AVM_DB_*`, nên `evaluate` tự ghi `test_b: skipped`. Không cần mật khẩu DB.
- Thời gian ước tính trên Colab CPU (2 vCPU): fetch ~5-10 phút, train ~30-60 phút, evaluate (có kiểm tra phường chưa thấy) ~30-60 phút. Runtime bị ngắt thì mất `/content` → dùng ô **Sao lưu lên Drive** sau mỗi giai đoạn lớn.
- Mỗi version train = một `run_id` mới (xem Bước 9). Cùng `run_id` mà đổi config → pipeline từ chối chạy.

## Bước 0. Chọn runtime

Menu **Runtime → Change runtime type**: chọn **CPU**, bật **High-RAM** nếu có (train 819k dòng cần ~6-8 GB RAM).

Không cần GPU: với 12 feature và ~820k dòng, LightGBM trên GPU chậm hơn CPU khoảng 6 lần (đã đo), nên pipeline chỉ chạy CPU.

In [ ]:
import os, platform, shutil, sys
print('python', sys.version.split()[0], '|', platform.platform())
print('cpu', os.cpu_count(), '| disk free GB', round(shutil.disk_usage('/content').free / 1e9, 1))
# Test B cần biến AVM_DB_*; xóa để chắc chắn nó bị bỏ qua.
for k in [k for k in os.environ if k.startswith('AVM_DB_')]:
    os.environ.pop(k)


## Bước 1. Gắn Google Drive (để lưu kết quả bền vững)

Bấm **Connect to Google Drive** khi được hỏi. Mọi thứ quan trọng sẽ được sao lưu vào `MyDrive/avm-train/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/avm-train'
os.makedirs(DRIVE, exist_ok=True)
print('Drive folder:', DRIVE)

## Bước 2. Lấy mã nguồn

Chọn **một** trong hai cách (đổi `CODE_SOURCE`):

**A. `zip` (khuyên dùng khi bạn có thay đổi chưa push):** trên máy Windows, tại thư mục repo `train-avm-opportunity`, chạy:
```
.venv\Scripts\python.exe -m zipfile -c avm-code.zip src configs tests pyproject.toml
```
rồi tải `avm-code.zip` lên `MyDrive/avm-train/` (kéo thả trên drive.google.com).

**B. `github`:** clone từ GitHub - chỉ có mã **đã push**. Repo private thì điền token vào `GITHUB_TOKEN` (tạo ở GitHub → Settings → Developer settings → Fine-grained token, quyền Contents: read). Không commit token vào notebook.

In [ ]:
CODE_SOURCE = 'zip'   # 'zip' hoặc 'github'
BRANCH = 'main'
GITHUB_TOKEN = ''     # chỉ cần khi repo private
REPO = '/content/train-avm-opportunity'

shutil.rmtree(REPO, ignore_errors=True)
if CODE_SOURCE == 'zip':
    os.makedirs(REPO)
    !unzip -q -o {DRIVE}/avm-code.zip -d {REPO}
else:
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone -q --branch {BRANCH} https://{auth}github.com/NguyenVanQThanh/train-avm-opportunity.git {REPO}
os.chdir(REPO)
!ls

## Bước 3. Cài thư viện

Cài package `avm-train` (lightgbm, pandas, pyarrow, scikit-learn, huggingface_hub, unidecode...) và `pytest`. Không cần `psycopg` vì không đọc DB.

In [ ]:
!pip install -q -e . pytest
import lightgbm, pandas, sklearn
print('lightgbm', lightgbm.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

## Bước 4. Tải file danh sách phường `wards.csv`

Stage `crosswalk` cần `data/wards.csv` (cột `code,name,province_code,province_name` - mã phường V-Nexus, **không phải PII**). Trên máy bạn file nằm ở `train-avm-opportunity/data/wards.csv` (3.321 dòng).

Cách 1: upload lên `MyDrive/avm-train/wards.csv` trước (ô dưới tự copy). Cách 2: ô dưới sẽ mở hộp thoại chọn file nếu không thấy trên Drive.

In [ ]:
os.makedirs('data', exist_ok=True)
if os.path.exists(f'{DRIVE}/wards.csv'):
    shutil.copy(f'{DRIVE}/wards.csv', 'data/wards.csv')
else:
    from google.colab import files
    up = files.upload()   # chọn wards.csv
    name = next(iter(up))
    shutil.move(name, 'data/wards.csv')
import pandas as pd
w = pd.read_csv('data/wards.csv', dtype=str)
print(w.shape, list(w.columns))
assert {'code', 'name', 'province_code'} <= set(w.columns), 'wards.csv sai cột'

## Bước 5. Kiểm tra nhanh mã nguồn (pytest)

48 test, chạy vài giây. Phải xanh hết trước khi tốn giờ train. (Nếu bạn zip không kèm `tests/` thì bỏ qua ô này.)

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## Bước 6. Cấu hình & hàm chạy stage

Config pin sẵn revision HF + commit crosswalk nên kết quả tái lập được. `run_stage` chạy từng stage, in log trực tiếp, lưu log vào `data/<run_id>/logs/` và dừng ngay khi stage lỗi.

Mã thoát `2` = **cổng dừng có chủ đích** (ví dụ khớp crosswalk < 95%, hoặc `evaluate` trượt hard gate) - đọc dòng `STOP:` để biết lý do.

In [ ]:
import subprocess, yaml
CONFIG = 'configs/v0_hf.yaml'
RUN_ID = yaml.safe_load(open(CONFIG, encoding='utf-8'))['run_id']
RUN_DIR = f'data/{RUN_ID}'
os.makedirs(f'{RUN_DIR}/logs', exist_ok=True)

def run_stage(stage, extra=(), fatal=True):
    cmd = [sys.executable, '-u', '-m', 'avm_train.cli', stage, '--config', CONFIG, *extra]
    env = {**os.environ, 'PYTHONUTF8': '1'}
    print('$', ' '.join(cmd), flush=True)
    with open(f'{RUN_DIR}/logs/{stage}.log', 'w', encoding='utf-8') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding='utf-8', env=env)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        rc = p.wait()
    print(f'--- {stage}: exit {rc}')
    if rc != 0 and fatal:
        raise RuntimeError(f'stage {stage} failed (exit {rc}); xem log ở trên')
    return rc

## Bước 7. Chạy pipeline từng stage

Mỗi stage đọc đầu ra của stage trước trong `data/<run_id>/`. Chạy tuần tự, **không bỏ qua stage nào**.

### 7.1 `fetch` - tải dữ liệu thô
Tải các shard của HF dataset (~1.7 GB, đúng revision đã pin) + file crosswalk từ GitHub (đúng commit đã pin). Giữ lại cột và khoảng ngày cần dùng.

In [ ]:
run_stage('fetch')

### 7.2 `crosswalk` - ghép phường cũ → phường mới → `wards.code` V-Nexus
Phải khớp ≥ 95% dòng có đủ tỉnh/quận/phường và ánh xạ 1-1 sang `wards.code`, nếu không dừng với `STOP:` (exit 2). Báo cáo ở `crosswalk/crosswalk_report.json`.

In [ ]:
run_stage('crosswalk', ['--wards-file', 'data/wards.csv'])

### 7.3 `clean` - làm sạch
Bỏ giá/diện tích ngoài khoảng, loại ptype ngoài phạm vi, bỏ trùng, bỏ outlier MAD. Báo cáo ở `clean/clean_report.json`. **Dữ liệu sạch** = `clean/listings.parquet`.

In [ ]:
run_stage('clean')

### 7.4 `split` - chia train / val / test_a theo thời gian
Train 2025-06 → 2026-01, val 2026-02 (dùng calibrate CQR), test_a 2026-03 (đánh giá cuối). Train có thêm fold theo phường.

In [ ]:
run_stage('split')

### 7.5 `features` - ma trận đặc trưng
Sinh `ward_price_ref` / `street_price_ref` (train mã hóa out-of-fold, chống rò nhãn) và `feature_schema.json` (bảng mã hóa đi kèm model).

In [ ]:
run_stage('features')

### 7.6 `baseline` - mốc phải vượt
Ước lượng đơn giản `ward_price_ref × diện tích`.

In [ ]:
run_stage('baseline')

### 7.7 `train` - 3 model LightGBM + CQR  (~30-60 phút)
Huấn luyện `point`, `q05`, `q95`, rồi calibrate khoảng tin cậy CQR trên val. Early stopping dùng 14 ngày cuối của train. Kết quả: `train/models/model_*.txt`, `train/conformal.json`, `train/train_report.json`.

In [ ]:
run_stage('train')

### Sao lưu giữa chừng lên Drive
Chạy ô này sau `train` để runtime bị ngắt cũng không phải train lại. (Bỏ `raw/` vì ~1.7 GB và tải lại được.)

In [ ]:
def backup():
    dst = f'{DRIVE}/{RUN_ID}'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(RUN_DIR, dst, ignore=shutil.ignore_patterns('raw'))
    print('đã sao lưu →', dst)
backup()

### 7.8 `evaluate` - bộ chỉ số + cổng chất lượng  (~30-60 phút)
Chạy MdAPE, PPE, COD, PRD, PRB, coverage khoảng tin cậy, cắt theo ptype / diện tích / tuần..., kiểm tra **phường chưa thấy** (refit 5 fold), so với baseline. Test B tự bỏ qua. Kết quả: `evaluate/metrics.json` có `verdict` PASS/FAIL.

Hard gate: Test A MdAPE ≤ 15% và PPE15 ≥ 50%. Trượt thì exit 2 (không phải lỗi mã) - ô dưới **không ném lỗi** để bạn vẫn xem và tải kết quả.

In [ ]:
rc = run_stage('evaluate', fatal=False)

## Bước 8. Xem kết quả

In [ ]:
import json
m = json.load(open(f'{RUN_DIR}/evaluate/metrics.json', encoding='utf-8'))
print('verdict:', m['verdict'], '| test_b:', m['test_b']['status'])
g = m['test_a']['global']; b = m['test_a']['baseline']
print(f"test_a  model   MdAPE={g['mdape']:.3f} PPE15={g['ppe15']:.3f} PRD={g['prd']:.3f} coverage={g['coverage']:.3f}")
print(f"test_a  baseline MdAPE={b['mdape']:.3f} PPE15={b['ppe15']:.3f}")
pd.DataFrame(m['checks'])[['check', 'level', 'status', 'value', 'limit']]

## Bước 8b. Đóng gói artifact cho BE (stage `export`)

Tạo `artifacts/<run_id>/`: 3 model, `conformal.json`, `feature_schema.json`, `metrics.json`, `crosswalk_report.json`, `golden.csv` (500 dòng kèm SHAP) và `manifest.json` (nguồn dữ liệu, seed, sha256 từng file, cờ `NON_PRODUCTION: true`).

- Chỉ export khi `evaluate` có verdict **PASS**. Trượt gate mà vẫn muốn xem thử thì đặt `FORCE_DRAFT = True` (manifest gắn `DRAFT: true`).
- Thư mục version là bất biến: chạy lại cùng `run_id` sẽ bị từ chối. Muốn export lại thì xóa `artifacts/<run_id>` hoặc đổi `run_id`.
- Trên Colab không có `.git` nên `manifest.code.commit` là `null`.

In [ ]:
FORCE_DRAFT = False
run_stage('export', ['--force-draft'] if FORCE_DRAFT else [])
!ls -la artifacts/{RUN_ID}

## Bước 9. Tải model và dữ liệu sạch về máy

Các gói riêng, mỗi gói tải về qua trình duyệt **và** lưu vào Drive:

| Gói | Nội dung | Dung lượng |
|---|---|---|
| `<run_id>-model.zip` | 3 model `model_{point,q05,q95}.txt`, `conformal.json`, `feature_schema.json`, `metrics.json`, `config.yaml`, các báo cáo | ~100 MB |
| `<run_id>-artifacts.zip` | Gói đăng ký cho BE: 3 model, `conformal.json`, `feature_schema.json`, `metrics.json`, `golden.csv`, `manifest.json` (có sau Bước 8b) | ~100 MB |
| `<run_id>-clean-data.zip` | `clean/listings.parquet`, `split/{train,val,test_a}.parquet`, `features/*.parquet` | ~70 MB |

Nếu trình duyệt chặn tải tự động, vào `drive.google.com → avm-train` để tải thủ công.

> Dữ liệu sạch xuất phát từ HF CC BY-NC 4.0: chỉ dùng nội bộ, **không đưa lên repo / không chia sẻ công khai**.

In [ ]:
import zipfile
from google.colab import files

def make_zip(name, items):
    path = f'/content/{name}'
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED) as z:
        for rel in items:
            p = pathlib.Path(RUN_DIR) / rel
            if p.is_dir():
                for f in p.rglob('*'):
                    if f.is_file():
                        z.write(f, f.relative_to(RUN_DIR))
            elif p.exists():
                z.write(p, p.relative_to(RUN_DIR))
            else:
                print('thiếu', rel)
    print(name, round(os.path.getsize(path) / 1e6, 1), 'MB')
    return path

import pathlib
model_zip = make_zip(f'{RUN_ID}-model.zip', [
    'train/models', 'train/conformal.json', 'train/train_report.json',
    'features/feature_schema.json', 'evaluate/metrics.json', 'baseline/baseline_metrics.json',
    'crosswalk/crosswalk_report.json', 'clean/clean_report.json', 'split/split_report.json',
    'config.yaml', 'logs'])
data_zip = make_zip(f'{RUN_ID}-clean-data.zip', ['clean/listings.parquet', 'split', 'features'])

art_dir = pathlib.Path('artifacts') / RUN_ID
zips = [model_zip, data_zip]
if art_dir.exists():
    art_zip = f'/content/{RUN_ID}-artifacts.zip'
    with zipfile.ZipFile(art_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in art_dir.iterdir():
            z.write(f, f'{RUN_ID}/{f.name}')
    print(art_zip, round(os.path.getsize(art_zip) / 1e6, 1), 'MB')
    zips.append(art_zip)

for z in zips:
    shutil.copy(z, DRIVE)          # bản lưu trên Drive
    files.download(z)              # tải về máy

## Bước 10. Train version mới (tùy chọn)

Mỗi version = một `run_id`, **không sửa config của run cũ** (pipeline từ chối). Cách làm:

1. Chạy ô dưới để tạo `configs/v1_hf.yaml` từ `v0_hf.yaml`, đổi `run_id`.
2. Sửa tham số muốn thử (ví dụ `model.num_boost_round: 6000` vì `point`/`q05` chạm trần 3000 vòng ở v0).
3. Đặt `CONFIG = 'configs/v1_hf.yaml'`, chạy lại Bước 6 → 9. Dữ liệu `raw/` của v0 không dùng chung, `fetch` sẽ tải lại (hoặc copy `data/v0-hf/raw` sang `data/v1-hf/raw` để bỏ qua `fetch`).

In [ ]:
NEW_RUN = 'v1-hf'
src = open('configs/v0_hf.yaml', encoding='utf-8').read().replace('run_id: v0-hf', f'run_id: {NEW_RUN}')
open('configs/v1_hf.yaml', 'w', encoding='utf-8').write(src)
print(open('configs/v1_hf.yaml', encoding='utf-8').read()[:300])

## Khôi phục sau khi runtime bị ngắt

Chạy lại Bước 0-3, rồi ô dưới để lấy `data/<run_id>/` từ Drive về (không gồm `raw/`; chỉ cần chạy lại `fetch` nếu muốn chạy lại từ đầu). Sau đó chạy tiếp từ stage còn dang dở.

In [ ]:
src_dir = f'{DRIVE}/{RUN_ID}'
if os.path.isdir(src_dir):
    shutil.copytree(src_dir, RUN_DIR, dirs_exist_ok=True)
    print('đã khôi phục', RUN_DIR, os.listdir(RUN_DIR))
else:
    print('chưa có bản sao lưu', src_dir)